In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedShuffleSplit, StratifiedKFold, cross_validate,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Set seed for reproducibility
seed = 67
rng = np.random.default_rng(seed) 

# Show all dataframe columns
pd.set_option("display.max_columns", None)

# Seaborn plot colors
sns.set_palette("viridis")

# EDA and Data Preparation

In [ ]:
hea_df = pd.read_csv("dataset10387_70.csv", index_col=0)

# Include Inv Omega parameter
hea_df['Inv Omega'] = hea_df['Enthalpy'].abs() / hea_df['Melting_point_K'] * hea_df['entropy']

# Drop Geo parameter
hea_df = hea_df.drop('Geo', axis=1)

hea_df.shape

In [ ]:
hea_df.head()

In [ ]:
hea_df.describe()

In [ ]:
hea_df.columns

In [ ]:
# Rename columns for clarity
renamed_cols = {
    'index': 'Alloy',
    'Num_el': 'Element Count',
    'Pauling_EN': 'Avg Electronegativity',
    'Melting_point_K': 'Melting Temp',
    'DFT_LDA_Etot': 'DFT Energy',
    'outer_shell_electrons': 'Outer Shell Electrons',
    'no_of_valence_electrons': 'Valence Electrons',
    'Atomic_radius_calculated': 'Atomic Radius',
    'Atomic_weight': 'Atomic Weight',
    'Pauling_EN_div': 'Electronegativity Diff',
    'entropy': 'Mixing Entropy',
    'Atomic_radius_calculated_dif': 'Atom Size Diff',
    'Enthalpy': 'Mixing Enthalpy',
    'Phase': 'Phase',
    'E_per_el': 'Energy per Element',
}

hea_df = hea_df.rename(columns=renamed_cols)
hea_df.head()

## Remove duplicates and inconsistensies

In [ ]:
# Remove alloy duplicates
deduped = hea_df.drop_duplicates()

# Drop alloys with inconsistent phases
inconsistent = deduped['Alloy'].value_counts()
inconsistent = deduped['Alloy'].isin(inconsistent[inconsistent > 1].index)
hea_df = deduped[~inconsistent].sort_index()

# Remove rows with NaN and inf values
hea_df = hea_df.replace([np.inf, -np.inf], np.nan).dropna()

hea_df.shape

## Phases Distribution

### Original Distribution

In [ ]:
order = hea_df["Phase"].value_counts().index
ax = sns.countplot(data=hea_df, x="Phase", order=order, hue="Phase", hue_order=order)
for container in ax.containers: ax.bar_label(container)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.show()

### Convert 'Phase' into multi-label binary columns

In [ ]:
# Split the phase strings into lists
phase_lists = hea_df['Phase'].str.split('+')

phase_cols = ['BCC', 'FCC', 'IM', 'AM']
phase_combs = ['BCC', 'FCC', 'IM', 'AM', 'BCC+FCC', 'FCC+IM', 'FCC+AM',
               'BCC+AM', 'BCC+IM', 'BCC+FCC+IM', 'BCC+FCC+AM']
has_phase_cols = ['has_BCC', 'has_FCC', 'has_IM', 'has_AM']

mlb = MultiLabelBinarizer(classes=phase_cols)
encoded = mlb.fit_transform(phase_lists)
encoded_df = pd.DataFrame(encoded, columns=has_phase_cols, index=hea_df.index)

# Attach to original df
hea_df = pd.concat([hea_df, encoded_df], axis=1)

In [ ]:
hea_df[['Alloy', 'Phase'] + has_phase_cols].head(10)

### Multi-label Distribution

In [ ]:
counts = hea_df[has_phase_cols].sum().sort_values(ascending=False)

ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index)
for container in ax.containers: ax.bar_label(container)

plt.ylabel('Count')
plt.xlabel('Phase')
plt.show()

## Alloying Elements Contribution

In [ ]:
hea_df.iloc[:, 2:55].columns

In [ ]:
elem_cols = hea_df.iloc[:, 2:55].columns

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)

for ax, phase, flag in zip(axes, phase_cols, has_phase_cols):
    # Per phase subset
    subset = hea_df.loc[hea_df[flag] == 1, elem_cols]

    # Get top 5 elements
    means = subset.mean().sort_values(ascending=False).head(5)

    sns.barplot(x=means.index, y=means.values, ax=ax, hue=means.values, legend=False)
    ax.set_title(f'{phase}  (n = {len(subset)})')
    ax.set_ylabel('Mean composition (at%)' if phase == 'BCC' else '')
    ax.set_xlabel('Element')

plt.tight_layout()
plt.show()

## Feature Correlation

In [ ]:
# Set features, X
feature_cols = ['VEC', 'Avg Electronegativity', 'Melting Temp', 'Outer Shell Electrons', 
                'Valence Electrons', 'Atomic Radius', 'Atomic Weight', 'Electronegativity Diff', 
                'Mixing Entropy', 'Atom Size Diff', 'Mixing Enthalpy', 'Inv Omega']

X = hea_df[feature_cols]

### Correlation Heatmaps

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), annot=True, fmt='.2f',vmin=-1, vmax=1, square=True, linewidths=0.5, cmap='coolwarm')
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Box Plots

In [ ]:
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
sns.boxplot(data=X_scaled, orient="h")
plt.show()

## Feature Selection via ExtraTreesClassifier

In [ ]:
# Set baseline and multilabel targets, y
y_base = hea_df["Phase"]
y_mulb = hea_df[has_phase_cols]

print("Features")
for i, col in enumerate(X.columns):
    print(f"{i}: {col}")

print("\nMulti-label Target")
for i, col in enumerate(y_mulb.columns):
    print(f"{i}: {col}")

In [ ]:
y_le = LabelEncoder().fit_transform(y_base)

# Fit Extra Trees
etc = ExtraTreesClassifier(
    n_estimators=500,
    max_features='sqrt',
    random_state=seed,
    n_jobs=-1,
    class_weight='balanced',
)

etc.fit(X, y_le)
importances = pd.Series(etc.feature_importances_, index=X.columns).sort_values(ascending=False)

# Plot importances
plt.figure(figsize=(9, 6))
ax = sns.barplot(x=importances.values, y=importances.index, hue=importances.values, legend=False)
ax.axvline(importances.mean(), color='r', ls='--', label='mean')
for container in ax.containers: ax.bar_label(container, fmt="%.4f", padding=3)
ax.set_xlabel('Mean decrease in impurity (Importance)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Select features
selector = SelectFromModel(etc, threshold='mean', prefit=True)
selected_features = X.columns[selector.get_support()].tolist()

print(f"Selected {len(selected_features)} of {len(X.columns)} features:")
print(selected_features)

X = X[selected_features]

## Outlier Removal via Isolation Forest

In [ ]:
def remove_outliers_per_class(X, y):
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)

    y_mask = np.ones(len(y_arr), dtype=bool)

    for phase in np.unique(y_arr):
        idx = np.where(y_arr == phase)[0]
        X_phase = X_arr[idx]

        iso = IsolationForest(
            contamination="auto",
            random_state=seed,
            n_estimators=200,
            n_jobs=-1,
        )
        preds = iso.fit_predict(X_phase)
        outlier_idx = idx[preds == -1]
        y_mask[outlier_idx] = False
        n_out = len(outlier_idx)

        print(f"{phase:<10} | {len(idx)-n_out:<6} | {n_out:<7} | {100*n_out/len(idx):.1f}%")

    return y_mask

print("Class      | Inlier | Outlier | % Removed")
print("-"*42)

y_mask = remove_outliers_per_class(X, y_base)
X = X.iloc[y_mask].reset_index(drop=True)
y_base = y_base.loc[y_mask].reset_index(drop=True)
y_mulb = y_mulb.loc[y_mask].reset_index(drop=True)

### After Outlier Removal

In [ ]:
X.shape

In [ ]:
# Phase Distribution (Baseline)

order = y_base.value_counts().index
ax = sns.countplot(x=y_base, order=order, hue=y_base, hue_order=order, legend=False)
for container in ax.containers: ax.bar_label(container)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.show()

In [ ]:
# Phase Distribution (Multilabel)

counts = y_mulb[has_phase_cols].sum().sort_values(ascending=False)
ax = sns.barplot(x=counts.index, y=counts.values, hue=counts.index)
for container in ax.containers: ax.bar_label(container)
plt.ylabel('Count')
plt.xlabel('Phase')
plt.show()

In [ ]:
# Correlation plot

plt.figure(figsize=(8, 6))
sns.heatmap(X.corr(), annot=True, fmt='.2f', vmin=-1, vmax=1, square=True, linewidths=0.5, cmap='coolwarm')
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Combine X and y temporarily
pair_df = X.copy()
pair_df['y_base'] = y_base

# Create pair plot with y as hue
pair_plot = sns.pairplot(pair_df, hue='y_base', diag_kind='kde', corner=True)

# Modify legend
handles, labels = pair_plot._legend_data.values(), pair_plot._legend_data.keys()
if pair_plot._legend is not None: pair_plot._legend.remove()
legend = pair_plot.fig.legend(
    handles=handles, 
    labels=labels,
    title='Phases',
    loc='upper right',
    bbox_to_anchor=(0.8, 0.8),
    fontsize='x-large',
    title_fontsize='x-large',
    frameon=True
)

plt.show()

In [ ]:
# Box plot
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
sns.boxplot(data=X_scaled, orient="h")
plt.show()

## Train-Test-Val Split

In [ ]:
# Split data to train-test sets
# Later CV will automatically get val set
test_ratio = 0.1

train_idx, test_idx = train_test_split(X.index, test_size=test_ratio, random_state=seed, stratify=y_base)

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_base_train, y_base_test = y_base.loc[train_idx], y_base.loc[test_idx]
y_mulb_train, y_mulb_test = y_mulb.loc[train_idx], y_mulb.loc[test_idx]

print(f"Split  | Train     | Test")
print("-" * 32)
print(f"X      | {str(X_train.shape):<9} | {str(X_test.shape)}")
print(f"y_base | {str(y_base_train.shape):<9} | {str(y_base_test.shape)}")
print(f"y_mulb | {str(y_mulb_train.shape):<9} | {str(y_mulb_test.shape)}")

# ML Training (Baseline)

In [ ]:
# TODO: Track training and predicting time

In [ ]:
# Double points of minority classes, keep majority untouched
# Preserves relative gaps and increases minority representation
minority_double_strategy = {
    'IM': 898,
    'FCC+IM': 278,
    'FCC+AM': 230,
    'BCC+IM': 180,
    'BCC+AM': 114,
    'BCC+FCC+IM': 62,
    'BCC+FCC+AM': 50,
}

smote = SMOTE(
    sampling_strategy=minority_double_strategy,
    random_state=seed,
)

# Create Scale -> SMOTE pipeline
def create_pipe(clf):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('smote', smote),
        ('clf', clf),
    ])

In [ ]:
"""def evaluate_model(model, X_train, y_train, X_test, y_test, name="model"):
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    def _metrics(y_true, y_pred):
        return {
            "accuracy":           accuracy_score(y_true, y_pred),
            "precision_macro":    precision_score(y_true, y_pred, average="macro",    zero_division=0),
            "recall_macro":       recall_score(y_true, y_pred,    average="macro",    zero_division=0),
            "f1_macro":           f1_score(y_true, y_pred,        average="macro",    zero_division=0),
            "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall_weighted":    recall_score(y_true, y_pred,    average="weighted", zero_division=0),
            "f1_weighted":        f1_score(y_true, y_pred,        average="weighted", zero_division=0),
        }

    trm = _metrics(y_train, y_train_pred)
    tem = _metrics(y_test,  y_test_pred)

    # Summary table
    print(name)
    print(f"{"Metrics":<18} | {'Train':<6} | {'Test'}")
    print("-" * 37)
    for m in trm:
        print(f"{m:<18} | {trm[m]:<6.4f} | {tem[m]:.4f}")
    print("")

    # Classification report
    print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))

    cm = confusion_matrix(y_test, y_test_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(cm, display_labels=phase_combs).plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion matrix - {name}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    return {
        "name":  name,
        "train": trm,
        "test":  tem,
        "cm":    cm,
        "y_train_pred": y_train_pred,
        "y_test_pred":  y_test_pred,
    }"""

## k-Nearest Neighbors

In [ ]:
"""# Default training

knn_pipe = create_pipe(KNeighborsClassifier())
knn_pipe.fit(X_train, y_base_train)
results = evaluate_model(
    knn_pipe, X_train, y_base_train, X_test, y_base_test,
    name="kNN (default)",
)"""

In [ ]:
# Default training

knn_pipe = create_pipe(KNeighborsClassifier())

In [ ]:
"""# Model tuning

param_dist = {
    "clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 51],
    "clf__weights":     ["uniform", "distance"],
    "clf__metric":      ["euclidean", "manhattan", "minkowski"],
    "clf__p":           [1, 2],           # only used by 'minkowski'
    "clf__leaf_size":   [20, 30, 40],
}

holdout = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)

search = RandomizedSearchCV(
    knn_pipe,
    param_distributions=param_dist,
    n_iter=100,
    scoring="f1_macro",
    cv=holdout,
    refit=True,
    n_jobs=-1,
    random_state=seed,
    verbose=1,
)
search.fit(X_train, y_base_train)

print(f"Best holdout f1_macro : {search.best_score_:.4f}")
print(f"Best params           : {search.best_params_}")

best_knn_pipe = search.best_estimator_"""

In [ ]:
"""skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

scoring = [
    "accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
]

def cv_report(model, X, y, name):
    cv = cross_validate(model, X, y, cv=skf, scoring=scoring,
                        n_jobs=-1, return_train_score=False)
    print(f"\n5-fold CV — {name}")
    print(f"{'metric':<22}{'mean':>10}{'std':>10}")
    print("-" * 42)
    for s in scoring:
        v = cv[f"test_{s}"]
        print(f"{s:<22}{v.mean():>10.4f}{v.std():>10.4f}")
    return cv

# (a) default hyperparameters, same pipeline (SMOTE x2 capped + scaler)
cv_default = cv_report(knn_pipe,      X_train, y_base_train, "kNN default")

# (b) best hyperparameters from search
cv_best    = cv_report(best_knn_pipe, X_train, y_base_train, "kNN best (holdout-tuned)")"""

In [ ]:
"""results_default = evaluate_model(
    knn_pipe,      X_train, y_base_train, X_test, y_base_test,
    name="kNN default (SMOTE x2 capped)",
)
results_best = evaluate_model(
    best_knn_pipe, X_train, y_base_train, X_test, y_base_test,
    name="kNN tuned (holdout-best)",
)"""

## Neural Network

## Support Vector Classifier

## Random Forest

# ML Training (Multilabel)